# Medical Necessity — Multi-Source Knowledge and Policy Learning

Extends the existing medical necessity work with two additional knowledge sources and a
reinforcement learning policy layer for point-of-order intervention.

Knowledge sources
1. CMS regulatory guidance (BPM10 10.2.1 / 10.2.3, 42 CFR 410.40 / 414.605 / 414.640)
2. Facility training content
3. Dave's summary from his discussion with Jen

Both new sources feed the rule-based scorer and the LLM extractor through one merged
knowledge base, so the two approaches stay directly comparable.

The policy layer trains on a synthetic golden dataset today and swaps to a real golden
dataset by changing a single assignment in section 9.

## 1. Configuration

In [ ]:
import os
import re
import json
import time
import random
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

WORKSPACE_BASE_URL = "https://adb-2790612761746757.17.azuredatabricks.net/serving-endpoints"
LLM_MODEL = "databricks-gpt-oss-120b"
LLM_TEMPERATURE = 0.1
LLM_MAX_TOKENS = 2500

OUTPUT_DIR = "/Workspace/Users/josh.smitherman@gmr.net/med_nec/data"
LOCAL_DIR = "/tmp"

SYNTHETIC_N = 1000
GOLDEN_TABLE = "prod-sandbox.josh_smitherman.med_nec_golden"

SCORE_NECESSARY_THRESHOLD = 3.0

AUTHORITY_RANK = {"regulatory": 3, "sme_guidance": 2, "operational_training": 1}

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

## 2. Knowledge source A — CMS regulatory guidance

Concept weights and axis assignment carry the source citation. `status` separates concepts
named in CMS guidance from those inferred from it; the five inferred concepts remain open
for SME validation.

In [ ]:
CMS_KNOWLEDGE = {
    "source_id": "CMS",
    "version": "2026-08",
    "authority": "regulatory",
    "provenance": "CMS Benefit Policy Manual Ch.10 and 42 CFR",
    "pending_ingest": False,
    "concepts": [
        {"concept": "bed_confined", "axis": "mobility", "weight": 2.0, "status": "named",
         "cite": "BPM10_10_2_1",
         "terms": ["bed confined", "bed-confined", "confined to bed", "bedbound", "bed bound",
                   "unable to get up from bed", "cannot get out of bed"]},
        {"concept": "cannot_ambulate", "axis": "mobility", "weight": 1.0, "status": "named",
         "cite": "BPM10_10_2_1",
         "terms": ["unable to ambulate", "cannot ambulate", "non-ambulatory", "nonambulatory",
                   "unable to walk", "cannot walk", "no weight bearing", "non weight bearing"]},
        {"concept": "cannot_sit_chair", "axis": "mobility", "weight": 1.0, "status": "named",
         "cite": "BPM10_10_2_1",
         "terms": ["unable to sit", "cannot sit in a chair", "cannot tolerate wheelchair",
                   "unable to tolerate sitting", "cannot sit upright"]},
        {"concept": "contraindicated_other_transport", "axis": "mobility", "weight": 1.0,
         "status": "named", "cite": "CFR_410_40",
         "terms": ["contraindicated", "other means of transport", "any other means",
                   "transport by other means would endanger", "would endanger the patient"]},
        {"concept": "paralysis_weakness", "axis": "mobility", "weight": 1.0, "status": "named",
         "cite": "BPM10_10_2_3",
         "terms": ["hemiparesis", "hemiplegia", "paraplegia", "quadriplegia", "tetraplegia",
                   "flaccid", "dense weakness", "paralysis"]},
        {"concept": "fracture_immobilization", "axis": "mobility", "weight": 1.0, "status": "named",
         "cite": "BPM10_10_2_3",
         "terms": ["unstable fracture", "spinal precautions", "cervical collar", "c-collar",
                   "traction", "external fixator", "pelvic binder", "log roll"]},
        {"concept": "cardiac_monitoring", "axis": "monitoring", "weight": 2.0, "status": "named",
         "cite": "CFR_414_605",
         "terms": ["cardiac monitor", "telemetry", "ekg monitoring", "ecg monitoring",
                   "continuous cardiac monitoring", "arrhythmia monitoring"]},
        {"concept": "iv_medication", "axis": "monitoring", "weight": 1.0, "status": "named",
         "cite": "CFR_414_605",
         "terms": ["iv drip", "iv infusion", "intravenous medication", "titratable drip",
                   "vasopressor", "heparin drip", "insulin drip", "continuous infusion"]},
        {"concept": "airway_ventilator", "axis": "monitoring", "weight": 2.0, "status": "named",
         "cite": "CFR_414_640",
         "terms": ["ventilator", "vented", "intubated", "endotracheal", "tracheostomy",
                   "trach", "bipap", "cpap", "airway management"]},
        {"concept": "suctioning", "axis": "monitoring", "weight": 1.0, "status": "named",
         "cite": "CFR_414_640",
         "terms": ["suction", "suctioning", "oral suction", "deep suction"]},
        {"concept": "oxygen", "axis": "monitoring", "weight": 0.3, "status": "inferred",
         "cite": "CFR_414_605",
         "terms": ["oxygen", "o2", "nasal cannula", "non-rebreather", "nrb", "liters per minute",
                   "lpm", "spo2", "supplemental o2"]},
        {"concept": "wound_ostomy", "axis": "monitoring", "weight": 0.3, "status": "inferred",
         "cite": "CFR_414_605",
         "terms": ["wound vac", "wound care", "ostomy", "colostomy", "ileostomy",
                   "pressure ulcer", "stage iii", "stage iv", "drain", "jp drain"]},
        {"concept": "isolation", "axis": "monitoring", "weight": 0.3, "status": "inferred",
         "cite": "CFR_414_605",
         "terms": ["isolation", "contact precautions", "droplet precautions",
                   "airborne precautions", "mrsa", "c diff", "cdiff", "vre"]},
        {"concept": "bariatric", "axis": "mobility", "weight": 0.3, "status": "inferred",
         "cite": "BPM10_10_2_3",
         "terms": ["bariatric", "morbid obesity", "morbidly obese", "over 350 lbs",
                   "over 400 lbs", "bariatric stretcher"]},
        {"concept": "behavioral", "axis": "monitoring", "weight": 0.3, "status": "inferred",
         "cite": "BPM10_10_2_3",
         "terms": ["combative", "agitated", "sitter", "one to one", "1:1", "restraints",
                   "psychiatric hold", "elopement risk", "behavioral health"]},
    ],
    "non_specific_phrases": [],
    "policy_rules": [
        {"rule_id": "cms_bed_trio",
         "statement": "Bed confinement requires all three: unable to get up from bed without "
                      "assistance, unable to ambulate, unable to sit in a chair or wheelchair.",
         "cite": "BPM10_10_2_1"},
        {"rule_id": "cms_bed_not_sole",
         "statement": "Bed confinement is one factor and is not by itself sufficient to "
                      "establish medical necessity.",
         "cite": "BPM10_10_2_1"},
    ],
}

## 3. Knowledge source B — facility training content

Replace the `concepts`, `non_specific_phrases`, and `policy_rules` entries with the content
extracted from the facility training material. The structure below is the contract the rest
of the notebook consumes; `pending_ingest` is asserted against in section 5 so a placeholder
source cannot be mistaken for a loaded one.

In [ ]:
FACILITY_TRAINING_KNOWLEDGE = {
    "source_id": "FACILITY_TRAINING",
    "version": "placeholder-0",
    "authority": "operational_training",
    "provenance": "Facility training content (not yet ingested)",
    "pending_ingest": True,
    "concepts": [
        {"concept": "cannot_ambulate", "axis": "mobility", "weight": 1.0, "status": "named",
         "cite": "FACILITY_TRAINING",
         "terms": ["max assist", "maximum assist", "two person assist", "2 person assist",
                   "total lift", "hoyer", "hoyer lift", "mechanical lift", "gait belt",
                   "dependent transfer"]},
        {"concept": "cannot_sit_chair", "axis": "mobility", "weight": 1.0, "status": "named",
         "cite": "FACILITY_TRAINING",
         "terms": ["cannot tolerate upright", "unable to maintain sitting balance",
                   "slides out of chair", "poor trunk control"]},
        {"concept": "contraindicated_other_transport", "axis": "mobility", "weight": 1.0,
         "status": "named", "cite": "FACILITY_TRAINING",
         "terms": ["no other means", "car transport unsafe", "family unable to transport safely",
                   "wheelchair van not appropriate"]},
        {"concept": "cardiac_monitoring", "axis": "monitoring", "weight": 2.0, "status": "named",
         "cite": "FACILITY_TRAINING",
         "terms": ["monitored bed", "step down monitoring", "requires monitoring en route"]},
        {"concept": "behavioral", "axis": "monitoring", "weight": 0.3, "status": "inferred",
         "cite": "FACILITY_TRAINING",
         "terms": ["requires sitter en route", "wanders", "exit seeking", "redirectable"]},
    ],
    "non_specific_phrases": [
        "needs ambulance", "needs transport", "per md order", "per physician order",
        "md order", "discharge", "transport to appointment", "for evaluation",
        "patient safety", "safety", "no other transportation", "routine transport",
        "hospital discharge", "return to facility", "per protocol", "as ordered",
    ],
    "policy_rules": [
        {"rule_id": "ft_state_why",
         "statement": "Documentation must state why the patient's condition makes other means "
                      "of transportation unsafe, not merely that transport was ordered.",
         "cite": "FACILITY_TRAINING"},
        {"rule_id": "ft_signature",
         "statement": "The certification statement requires ordering practitioner signature, "
                      "date, and the reason for transport tied to the patient's condition.",
         "cite": "FACILITY_TRAINING"},
        {"rule_id": "ft_specific_over_generic",
         "statement": "A specific clinical finding replaces a generic phrase; generic phrases "
                      "alone do not establish either axis.",
         "cite": "FACILITY_TRAINING"},
    ],
}

## 4. Knowledge source C — Dave's summary from his discussion with Jen

SME guidance ranks above facility training and below CMS, except on concepts CMS does not
name, where the SME read wins. Every override is logged with provenance so Jen and Michelle
can audit exactly which source set each weight.

In [ ]:
DAVE_JEN_KNOWLEDGE = {
    "source_id": "DAVE_JEN",
    "version": "placeholder-0",
    "authority": "sme_guidance",
    "provenance": "Dave's summary of discussion with Jen (not yet ingested)",
    "pending_ingest": True,
    "concepts": [
        {"concept": "oxygen", "axis": "monitoring", "weight": 0.3, "status": "inferred",
         "cite": "DAVE_JEN",
         "terms": ["continuous o2", "o2 dependent", "oxygen dependent"]},
        {"concept": "behavioral", "axis": "monitoring", "weight": 0.5, "status": "inferred",
         "cite": "DAVE_JEN",
         "terms": ["requires restraint en route", "chemical restraint", "danger to self"]},
        {"concept": "iv_medication", "axis": "monitoring", "weight": 1.0, "status": "named",
         "cite": "DAVE_JEN",
         "terms": ["iv access maintained", "medication due en route", "pca pump"]},
    ],
    "non_specific_phrases": ["stable for transport", "no acute distress"],
    "policy_rules": [
        {"rule_id": "sme_oxygen_not_sole",
         "statement": "Oxygen alone does not establish the monitoring axis; it supports an "
                      "otherwise documented monitoring need.",
         "cite": "DAVE_JEN"},
        {"rule_id": "sme_behavioral_review",
         "statement": "Behavioral is the inferred concept most likely to be reclassified and is "
                      "the highest priority SME review item.",
         "cite": "DAVE_JEN"},
        {"rule_id": "sme_indeterminate_routes",
         "statement": "Indeterminate documentation routes to medical necessity review rather "
                      "than auto-denial.",
         "cite": "DAVE_JEN"},
        {"rule_id": "sme_denial_not_proof",
         "statement": "A denial is not proof of a medical necessity failure; only one denial "
                      "reason is captured per claim.",
         "cite": "DAVE_JEN"},
    ],
}

KNOWLEDGE_SOURCES = [CMS_KNOWLEDGE, FACILITY_TRAINING_KNOWLEDGE, DAVE_JEN_KNOWLEDGE]

## 5. Knowledge merge and provenance

Terms union across all three sources. Weight and status resolve by authority rank, with
every override recorded. One exception: SME guidance may refine a concept whose status is
`inferred`, because an inferred concept is not stated in CMS text and the SME read is the
better authority on it. Behavioral is the concept this exception exists for. Non-specific phrases and policy rules concatenate.

In [ ]:
@dataclass
class MergedConcept:
    concept: str
    axis: str
    weight: float
    status: str
    terms: List[str]
    weight_source: str
    cites: List[str]
    term_sources: Dict[str, List[str]]


def merge_knowledge(sources: List[Dict[str, Any]]):
    ordered = sorted(sources, key=lambda s: AUTHORITY_RANK[s["authority"]], reverse=True)
    merged: Dict[str, MergedConcept] = {}
    overrides: List[Dict[str, Any]] = []

    for src in ordered:
        rank = AUTHORITY_RANK[src["authority"]]
        for c in src["concepts"]:
            key = c["concept"]
            if key not in merged:
                merged[key] = MergedConcept(
                    concept=key, axis=c["axis"], weight=float(c["weight"]),
                    status=c["status"], terms=list(dict.fromkeys(c["terms"])),
                    weight_source=src["source_id"], cites=[c["cite"]],
                    term_sources={src["source_id"]: list(c["terms"])},
                )
            else:
                m = merged[key]
                new_terms = [t for t in c["terms"] if t not in m.terms]
                m.terms.extend(new_terms)
                m.term_sources.setdefault(src["source_id"], []).extend(c["terms"])
                if c["cite"] not in m.cites:
                    m.cites.append(c["cite"])
                incumbent_rank = AUTHORITY_RANK[
                    next(s["authority"] for s in sources if s["source_id"] == m.weight_source)]
                if float(c["weight"]) != m.weight or c["status"] != m.status:
                    overrides.append({
                        "concept": key,
                        "proposed_by": src["source_id"],
                        "proposed_weight": float(c["weight"]),
                        "proposed_status": c["status"],
                        "held_by": m.weight_source,
                        "held_weight": m.weight,
                        "held_status": m.status,
                        "applied": bool(rank > incumbent_rank
                                        or (src["authority"] == "sme_guidance"
                                            and m.status == "inferred")),
                    })
                    refines_inferred = (src["authority"] == "sme_guidance"
                                        and m.status == "inferred")
                    if rank > incumbent_rank or refines_inferred:
                        m.weight = float(c["weight"])
                        m.status = c["status"]
                        m.weight_source = src["source_id"]

    non_specific, rules = [], []
    for src in sources:
        for p in src.get("non_specific_phrases", []):
            if p not in non_specific:
                non_specific.append(p)
        for r in src.get("policy_rules", []):
            rules.append({**r, "source_id": src["source_id"]})

    kb = {
        "concepts": merged,
        "non_specific_phrases": non_specific,
        "policy_rules": rules,
        "overrides": overrides,
        "sources": [{"source_id": s["source_id"], "version": s["version"],
                     "authority": s["authority"], "provenance": s["provenance"],
                     "pending_ingest": s["pending_ingest"],
                     "n_concepts": len(s["concepts"]),
                     "n_rules": len(s.get("policy_rules", []))} for s in sources],
    }
    return kb


KB = merge_knowledge(KNOWLEDGE_SOURCES)

source_frame = pd.DataFrame(KB["sources"])
concept_frame = pd.DataFrame([{
    "concept": c.concept, "axis": c.axis, "weight": c.weight, "status": c.status,
    "weight_source": c.weight_source, "n_terms": len(c.terms),
    "cites": "|".join(c.cites), "term_sources": "|".join(sorted(c.term_sources))
} for c in KB["concepts"].values()]).sort_values(["axis", "concept"]).reset_index(drop=True)
override_frame = pd.DataFrame(KB["overrides"])

print(source_frame.to_string(index=False))
print()
print(concept_frame.to_string(index=False))
print()
print("Weight and status conflicts")
print(override_frame.to_string(index=False) if len(override_frame) else "none")
print()
pending = [s["source_id"] for s in KB["sources"] if s["pending_ingest"]]
print("PENDING_INGEST:", pending if pending else "none")

## 6. Rule-based scorer

`total_score = mobility_score + monitoring_score + named_score`. Bucket rule is unchanged:
`total_score == 0` is not_necessary, `total_score >= 3` with a named concept is necessary,
everything else is indeterminate. The non-specific phrase list contributed by the facility
training source drives a `vague_only` flag rather than a score penalty, so the arithmetic
stays verifiable.

In [ ]:
def build_matchers(kb):
    matchers = {}
    for c in kb["concepts"].values():
        parts = [re.escape(t) for t in sorted(c.terms, key=len, reverse=True)]
        matchers[c.concept] = re.compile(r"(?<![a-z0-9])(" + "|".join(parts) + r")(?![a-z0-9])")
    phrases = [re.escape(p) for p in sorted(kb["non_specific_phrases"], key=len, reverse=True)]
    vague = re.compile(r"(" + "|".join(phrases) + r")") if phrases else None
    return matchers, vague


MATCHERS, VAGUE_MATCHER = build_matchers(KB)


def score_text(text: str, kb=KB, matchers=MATCHERS, vague=VAGUE_MATCHER) -> Dict[str, Any]:
    raw = (text or "").strip()
    low = raw.lower()
    out = {
        "has_text": int(bool(low)),
        "text_len": len(raw),
        "mobility_score": 0.0,
        "monitoring_score": 0.0,
        "named_score": 0.0,
        "total_score": 0.0,
        "has_named_concept": 0,
        "vague_only": 0,
        "hits": [],
        "evidence": {},
    }
    if not low:
        out["determination"] = "not_necessary"
        return out

    for concept, rx in matchers.items():
        m = rx.search(low)
        if not m:
            continue
        c = kb["concepts"][concept]
        out["hits"].append(concept)
        out["evidence"][concept] = m.group(0)
        if c.axis == "mobility":
            out["mobility_score"] += c.weight
        else:
            out["monitoring_score"] += c.weight
        if c.status == "named":
            out["has_named_concept"] = 1

    out["named_score"] = 1.0 if out["has_named_concept"] else 0.0
    out["total_score"] = out["mobility_score"] + out["monitoring_score"] + out["named_score"]

    if vague is not None and vague.search(low) and not out["hits"]:
        out["vague_only"] = 1

    if out["total_score"] == 0.0:
        out["determination"] = "not_necessary"
    elif out["total_score"] >= SCORE_NECESSARY_THRESHOLD and out["has_named_concept"]:
        out["determination"] = "necessary"
    else:
        out["determination"] = "indeterminate"
    return out


def missing_axes_from_score(s: Dict[str, Any]) -> List[str]:
    gaps = []
    if s["mobility_score"] <= 0:
        gaps.append("mobility")
    if s["monitoring_score"] <= 0:
        gaps.append("monitoring")
    return gaps


for sample in [
    "Pt bed confined, unable to ambulate, requires two person assist. Continuous cardiac monitoring en route.",
    "Needs ambulance per MD order for discharge.",
    "Patient on 2L nasal cannula, contact precautions for MRSA.",
    "",
]:
    r = score_text(sample)
    print(f"{r['determination']:14s} total={r['total_score']:.1f} "
          f"mob={r['mobility_score']:.1f} mon={r['monitoring_score']:.1f} "
          f"named={r['named_score']:.1f} vague_only={r['vague_only']} hits={r['hits']}")

## 7. Model client

Same invocation as the nurse navigation notebook: the OpenAI SDK pointed at the Databricks
serving endpoint with the notebook context token. When this moves behind an endpoint the
token becomes a service principal token and nothing else in the cell changes.

In [ ]:
from openai import OpenAI

DATABRICKS_TOKEN = (
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
    if "dbutils" in dir() else os.environ.get("DATABRICKS_TOKEN", "")
)
client = OpenAI(api_key=DATABRICKS_TOKEN or "token-not-set",
                base_url=WORKSPACE_BASE_URL)


def llm_call(system_prompt, user_prompt, max_tokens=2500):
    resp = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "system", "content": system_prompt},
                  {"role": "user", "content": user_prompt}],
        temperature=0.1, max_tokens=max_tokens)
    content = resp.choices[0].message.content
    if isinstance(content, list):
        for item in content:
            if isinstance(item, dict) and item.get("type") == "text":
                return item.get("text", "")
        return json.dumps(content)
    return content


LLM_AVAILABLE = bool(DATABRICKS_TOKEN)
print("token present:", LLM_AVAILABLE, "| model:", LLM_MODEL)

## 8. Extraction prompt

The concept list, the phrasing seen in practice, the non-specific phrase list, and the
policy rules are all compiled from the merged knowledge base, so the facility training
content and Dave's summary reach the model without a separate prompt edit. The model returns
facts only; the label is assigned in section 9.

In [ ]:
def compile_prompt(kb):
    concept_lines, keys = [], []
    for c in sorted(kb["concepts"].values(), key=lambda x: (x.axis, x.concept)):
        keys.append(c.concept)
        concept_lines.append(
            f'  "{c.concept}": boolean   ({c.axis} axis, {c.status}; '
            f'phrasing seen in practice: {", ".join(c.terms[:8])})')
    rule_lines = [f"- [{r['source_id']}] {r['statement']}" for r in kb["policy_rules"]]
    schema = ",\n".join(f'    "{k}": boolean' for k in keys)
    return keys, f"""
You are a clinical documentation extraction assistant for non-emergent ground ambulance
transport orders. Extract only facts stated in the "Clinical Data" text. Do not guess.
Handle negations. Do not decide medical necessity.
For every boolean set to true you MUST provide a verbatim evidence quote.

Concepts to extract:
{chr(10).join(concept_lines)}

Guidance that governs extraction:
{chr(10).join(rule_lines)}

These phrases are non-specific and never on their own establish a concept:
{", ".join(kb["non_specific_phrases"])}

OUTPUT valid JSON only, all keys present, booleans never null:
{{
  "concepts": {{
{schema}
  }},
  "non_specific_only": boolean,
  "evidence": [{{"field": string, "value": boolean, "quote": string}}]
}}
Set a concept true only when the text supports it and you can quote the text. If the text is
empty or contains only non-specific phrases, set every concept false and non_specific_only true.
Now extract from the following Clinical Data."""


CONCEPT_KEYS, MED_NEC_PROMPT = compile_prompt(KB)
print(MED_NEC_PROMPT[:1600])
print("...")
print("prompt characters:", len(MED_NEC_PROMPT), "| concepts:", len(CONCEPT_KEYS))

## 9. Validator and deterministic judge

`valid_json` is the same quality gate used in the nurse navigation notebook: a response that
does not parse or is missing a required section counts as a failed extraction rather than a
silent default. The judge applies the same weights and cutoffs as the rule scorer, so any
difference between the two paths is attributable to extraction and not to scoring.

In [ ]:
def valid_json(raw, keys=CONCEPT_KEYS):
    try:
        obj = json.loads(raw)
    except Exception:
        return None
    if "concepts" not in obj or not isinstance(obj["concepts"], dict):
        return None
    quotes = {}
    for e in obj.get("evidence", []) or []:
        if isinstance(e, dict) and e.get("value"):
            quotes[e.get("field")] = str(e.get("quote", "") or "")
    concepts = {}
    for k in keys:
        flag = bool(obj["concepts"].get(k, False))
        concepts[k] = {"documented": flag and bool(quotes.get(k, "")),
                       "evidence": quotes.get(k, "")}
    return {"concepts": concepts, "non_specific_only": bool(obj.get("non_specific_only", False))}


def judge(extraction, kb=KB):
    mobility = monitoring = 0.0
    named = 0
    hits, evidence = [], {}
    for name, node in extraction["concepts"].items():
        if not node["documented"]:
            continue
        c = kb["concepts"][name]
        hits.append(name)
        evidence[name] = node["evidence"]
        if c.axis == "mobility":
            mobility += c.weight
        else:
            monitoring += c.weight
        if c.status == "named":
            named = 1
    named_score = 1.0 if named else 0.0
    total = mobility + monitoring + named_score
    if total == 0.0:
        determination = "not_necessary"
    elif total >= SCORE_NECESSARY_THRESHOLD and named:
        determination = "necessary"
    else:
        determination = "indeterminate"
    return {
        "llm_mobility_score": mobility,
        "llm_monitoring_score": monitoring,
        "llm_named_score": named_score,
        "llm_total_score": total,
        "llm_has_named_concept": named,
        "llm_vague_only": int(bool(extraction.get("non_specific_only")) and not hits),
        "llm_determination": determination,
        "llm_hits": hits,
        "llm_evidence": evidence,
    }


def extract_offline(text, kb=KB, noise=0.06, rng=None):
    rng = rng or random.Random(SEED)
    low = (text or "").lower()
    concepts = {}
    for name, c in kb["concepts"].items():
        hit = any(t in low for t in c.terms)
        if rng.random() < noise:
            hit = not hit
        ev = next((t for t in c.terms if t in low), "") if hit else ""
        concepts[name] = {"documented": bool(hit and (ev or rng.random() < 0.5)),
                          "evidence": ev}
    vague = bool(VAGUE_MATCHER.search(low)) if VAGUE_MATCHER else False
    return {"concepts": concepts,
            "non_specific_only": vague and not any(v["documented"] for v in concepts.values())}

## 10. Golden dataset interface

Every downstream section reads `GOLDEN_SOURCE.load()` and nothing else. Moving from the
synthetic prototype to the real golden dataset is one assignment at the bottom of this cell,
provided the returned frame satisfies `GOLDEN_SCHEMA`.

In [ ]:
GOLDEN_SCHEMA = {
    "order_id": "string",
    "clinical_text": "string",
    "facility": "string",
    "level_of_service": "string",
    "payer_type": "string",
    "gold_determination": "string",
    "gold_missing_axes": "string",
    "would_deny": "int",
    "gold_source": "string",
}

VALID_DETERMINATIONS = {"necessary", "indeterminate", "not_necessary"}
VALID_AXES = {"", "mobility", "monitoring", "mobility|monitoring"}


def validate_golden(df: pd.DataFrame) -> pd.DataFrame:
    missing = [c for c in GOLDEN_SCHEMA if c not in df.columns]
    if missing:
        raise ValueError(f"golden dataset missing columns: {missing}")
    bad_det = set(df["gold_determination"]) - VALID_DETERMINATIONS
    if bad_det:
        raise ValueError(f"unexpected gold_determination values: {bad_det}")
    bad_axes = set(df["gold_missing_axes"].fillna("")) - VALID_AXES
    if bad_axes:
        raise ValueError(f"unexpected gold_missing_axes values: {bad_axes}")
    if not set(df["would_deny"].unique()) <= {0, 1}:
        raise ValueError("would_deny must be 0 or 1")
    if df["order_id"].duplicated().any():
        raise ValueError("order_id is not unique")
    return df[list(GOLDEN_SCHEMA)].copy()


class GoldenSource:
    name = "base"

    def load(self) -> pd.DataFrame:
        raise NotImplementedError


class DeltaTableGoldenSource(GoldenSource):
    name = "delta_table"

    def __init__(self, table: str):
        self.table = table

    def load(self) -> pd.DataFrame:
        df = spark.table(self.table).toPandas()
        return validate_golden(df)

## 11. Synthetic golden dataset

Archetypes are generated from the merged term lists, so text produced here exercises the
facility training vocabulary and the SME-adjusted concepts as well as the CMS terms. Labels
are assigned by construction, not by scoring, which keeps the evaluation in section 12
independent of both approaches under test.

In [ ]:
FACILITIES = ["THR Presbyterian Dallas", "THR Harris Fort Worth", "MUSC Main",
              "MUSC Ashley River", "THR Plano"]
LEVELS = ["BLS", "ALS", "BLS", "BLS", "ALS"]
PAYERS = ["Medicare", "Medicare", "Medicare Advantage", "Commercial", "Medicaid"]

FILLER = ["Patient stable at time of order.", "Receiving facility notified.",
          "Family updated.", "Report called.", "Transport requested for continued care."]


def _terms(concept: str, rng: random.Random, k: int = 1) -> List[str]:
    c = KB["concepts"][concept]
    return rng.sample(c.terms, min(k, len(c.terms)))


def _sentence(concept: str, rng: random.Random) -> str:
    t = _terms(concept, rng, 1)[0]
    frames = ["Patient {t}.", "Documented {t} at time of order.",
              "Nursing reports {t}.", "{t} noted this shift."]
    return rng.choice(frames).format(t=t).capitalize()


ARCHETYPES = [
    ("both_axes", 0.22, ["bed_confined", "cannot_ambulate", "cardiac_monitoring"],
     "necessary", ""),
    ("mobility_strong", 0.12,
     ["bed_confined", "cannot_ambulate", "cannot_sit_chair", "contraindicated_other_transport"],
     "necessary", "monitoring"),
    ("monitoring_strong", 0.10, ["airway_ventilator", "suctioning", "iv_medication"],
     "necessary", "mobility"),
    ("mobility_only_partial", 0.12, ["cannot_ambulate"], "indeterminate", "monitoring"),
    ("monitoring_only_partial", 0.10, ["oxygen", "iv_medication"], "indeterminate", "mobility"),
    ("inferred_only", 0.10, ["oxygen", "isolation"], "indeterminate", "mobility|monitoring"),
    ("behavioral_only", 0.06, ["behavioral"], "indeterminate", "mobility"),
    ("training_vocab", 0.08, ["cannot_ambulate", "cannot_sit_chair"],
     "indeterminate", "monitoring"),
    ("vague_only", 0.06, [], "not_necessary", "mobility|monitoring"),
    ("empty", 0.04, [], "not_necessary", "mobility|monitoring"),
]


class SyntheticGoldenSource(GoldenSource):
    name = "synthetic"

    def __init__(self, n: int = SYNTHETIC_N, seed: int = SEED):
        self.n = n
        self.seed = seed

    def load(self) -> pd.DataFrame:
        rng = random.Random(self.seed)
        names = [a[0] for a in ARCHETYPES]
        weights = [a[1] for a in ARCHETYPES]
        rows = []
        for i in range(self.n):
            arch = rng.choices(ARCHETYPES, weights=weights, k=1)[0]
            arch_name, _, concepts, gold, missing = arch
            if arch_name == "empty":
                text = ""
            elif arch_name == "vague_only":
                text = " ".join(rng.sample(KB["non_specific_phrases"],
                                           min(3, len(KB["non_specific_phrases"]))))
                text = text.capitalize() + "."
            else:
                parts = [_sentence(c, rng) for c in concepts]
                if rng.random() < 0.5:
                    parts.append(rng.choice(FILLER))
                if rng.random() < 0.25:
                    parts.append(rng.choice(KB["non_specific_phrases"]).capitalize() + ".")
                rng.shuffle(parts)
                text = " ".join(parts)
            idx = rng.randrange(len(FACILITIES))
            rows.append({
                "order_id": f"SYN-{100000 + i}",
                "clinical_text": text,
                "facility": FACILITIES[idx],
                "level_of_service": LEVELS[idx],
                "payer_type": rng.choice(PAYERS),
                "gold_determination": gold,
                "gold_missing_axes": missing,
                "would_deny": 0 if gold == "necessary" else 1,
                "gold_source": f"synthetic:{arch_name}",
            })
        return validate_golden(pd.DataFrame(rows))


GOLDEN_SOURCE = SyntheticGoldenSource(n=SYNTHETIC_N, seed=SEED)

golden = GOLDEN_SOURCE.load()
print("golden source:", GOLDEN_SOURCE.name, "rows:", len(golden))
print(golden["gold_determination"].value_counts().to_string())
print()
print(golden["gold_source"].value_counts().to_string())
print()
print(golden[["order_id", "gold_determination", "gold_missing_axes", "clinical_text"]]
      .head(6).to_string(index=False))

## 12. Scoring both approaches against the golden dataset

Unsafe pass is an order the golden label marks deniable that the approach called necessary.
It is reported separately from overall accuracy because the two failure modes carry very
different cost.

In [ ]:
rows = []
for r in golden.to_dict("records"):
    prompt = (f"order_id: {r['order_id']}\n"
              f"level_of_service: {r['level_of_service']}\n"
              f"payer_type: {r['payer_type']}\n\n"
              f"Clinical Data:\n{r['clinical_text']}")
    obj = None
    if LLM_AVAILABLE:
        try:
            obj = valid_json(llm_call(MED_NEC_PROMPT, prompt))
        except Exception:
            obj = None
    rows.append({"order_id": r["order_id"], "ok": obj is not None, "obj": obj})

ext = pd.DataFrame(rows)
print(f"Valid extractions: {ext['ok'].mean()*100:.1f}%  ({ext['ok'].sum()} of {len(ext)})")

rng_extract = random.Random(SEED)
fallback = 0
extractions = {}
for r, e in zip(golden.to_dict("records"), rows):
    if e["ok"]:
        extractions[r["order_id"]] = (e["obj"], "llm")
    else:
        fallback += 1
        extractions[r["order_id"]] = (extract_offline(r["clinical_text"], rng=rng_extract),
                                      "offline_simulator")
print("rows falling back to the offline simulator:", fallback)

records = []
for row in golden.to_dict("records"):
    s = score_text(row["clinical_text"])
    extraction, mode = extractions[row["order_id"]]
    j = judge(extraction)
    rec = dict(row)
    rec.update({
        "mobility_score": s["mobility_score"],
        "monitoring_score": s["monitoring_score"],
        "named_score": s["named_score"],
        "total_score": s["total_score"],
        "has_named_concept": s["has_named_concept"],
        "vague_only": s["vague_only"],
        "has_text": s["has_text"],
        "text_len": s["text_len"],
        "rule_determination": s["determination"],
        "rule_hits": "|".join(s["hits"]),
        "extraction_mode": mode,
    })
    rec.update({k: v for k, v in j.items() if k not in ("llm_hits", "llm_evidence")})
    rec["llm_hits"] = "|".join(j["llm_hits"])
    records.append(rec)

scored = pd.DataFrame(records)
print("extraction mode:", scored["extraction_mode"].value_counts().to_dict())


def approach_metrics(df: pd.DataFrame, pred_col: str, label: str) -> Dict[str, Any]:
    correct = (df[pred_col] == df["gold_determination"]).mean()
    unsafe = ((df[pred_col] == "necessary") & (df["would_deny"] == 1)).mean()
    over_flag = ((df[pred_col] != "necessary") & (df["would_deny"] == 0)).mean()
    return {"approach": label, "accuracy": round(correct, 4),
            "unsafe_pass_rate": round(unsafe, 4),
            "unnecessary_friction_rate": round(over_flag, 4)}


metrics = pd.DataFrame([
    approach_metrics(scored, "rule_determination", "rule_based"),
    approach_metrics(scored, "llm_determination", "llm_extract_then_judge"),
])
print()
print(metrics.to_string(index=False))
print()
print("Rule determination against gold")
print(pd.crosstab(scored["gold_determination"], scored["rule_determination"]).to_string())
print()
print("LLM determination against gold")
print(pd.crosstab(scored["gold_determination"], scored["llm_determination"]).to_string())
print()
print("Rule against LLM")
print(pd.crosstab(scored["rule_determination"], scored["llm_determination"]).to_string())

## 13. Action space and reward function

The policy decides what happens at order entry, not whether the order is medically
necessary. That determination stays with the deterministic judge; the policy chooses the
intervention.

Reward values are the tunable part of the design and are stated in one table so Jen and
Michelle can set them directly. Once the denial extract from Vic and Robin lands,
`REWARD_TABLE` is replaced by realized denial cost and review labor cost per order.

In [ ]:
ACTIONS = ["accept", "prompt_mobility", "prompt_monitoring", "route_review"]
ACTION_INDEX = {a: i for i, a in enumerate(ACTIONS)}

REWARD_TABLE = {
    "accept_clean": 1.0,
    "accept_deniable": -6.0,
    "prompt_on_target": 3.0,
    "prompt_off_target": -2.0,
    "prompt_unneeded": -0.5,
    "review_deniable": 1.0,
    "review_clean": -1.0,
}


def reward(action: str, row: Dict[str, Any], table=REWARD_TABLE) -> float:
    deniable = int(row["would_deny"]) == 1
    missing = set(str(row.get("gold_missing_axes") or "").split("|")) - {""}
    if action == "accept":
        return table["accept_deniable"] if deniable else table["accept_clean"]
    if action == "route_review":
        return table["review_deniable"] if deniable else table["review_clean"]
    axis = action.replace("prompt_", "")
    if not deniable:
        return table["prompt_unneeded"]
    return table["prompt_on_target"] if axis in missing else table["prompt_off_target"]


demo = golden.to_dict("records")[0]
print({a: reward(a, demo) for a in ACTIONS}, "|", demo["gold_determination"],
      "|", demo["gold_missing_axes"])

## 14. Context features and contextual bandit policy

Context is built from both approaches plus the order attributes, so the policy can learn
where the rule path and the extraction path disagree. LinUCB is used rather than a deep
policy because the feature space is small, the update is closed form, and the per-arm
coefficients are readable by an SME.

In [ ]:
FACILITY_INDEX = {f: i for i, f in enumerate(sorted(golden["facility"].unique()))}
PAYER_INDEX = {p: i for i, p in enumerate(sorted(golden["payer_type"].unique()))}

FEATURE_NAMES = [
    "bias", "mobility_score", "monitoring_score", "named_score", "vague_only", "has_text",
    "text_len", "llm_mobility_score", "llm_monitoring_score", "llm_named_score",
    "llm_vague_only", "rule_necessary", "rule_indeterminate", "rule_not_necessary",
    "llm_necessary", "llm_indeterminate", "llm_not_necessary", "paths_agree",
    "mobility_gap", "monitoring_gap", "is_als", "is_medicare",
]


def featurize(row: Dict[str, Any]) -> np.ndarray:
    mob = float(row["mobility_score"])
    mon = float(row["monitoring_score"])
    lmob = float(row["llm_mobility_score"])
    lmon = float(row["llm_monitoring_score"])
    v = [
        1.0,
        min(mob, 6.0) / 6.0,
        min(mon, 6.0) / 6.0,
        float(row["named_score"]),
        float(row["vague_only"]),
        float(row["has_text"]),
        min(float(row["text_len"]), 400.0) / 400.0,
        min(lmob, 6.0) / 6.0,
        min(lmon, 6.0) / 6.0,
        float(row["llm_named_score"]),
        float(row["llm_vague_only"]),
        float(row["rule_determination"] == "necessary"),
        float(row["rule_determination"] == "indeterminate"),
        float(row["rule_determination"] == "not_necessary"),
        float(row["llm_determination"] == "necessary"),
        float(row["llm_determination"] == "indeterminate"),
        float(row["llm_determination"] == "not_necessary"),
        float(row["rule_determination"] == row["llm_determination"]),
        float(mob <= 0 and lmob <= 0),
        float(mon <= 0 and lmon <= 0),
        float(row["level_of_service"] == "ALS"),
        float(str(row["payer_type"]).startswith("Medicare")),
    ]
    return np.array(v, dtype=float)


class LinUCBPolicy:
    def __init__(self, n_actions: int, n_features: int, alpha: float = 0.6, l2: float = 1.0):
        self.n_actions = n_actions
        self.alpha = alpha
        self.A = [np.eye(n_features) * l2 for _ in range(n_actions)]
        self.b = [np.zeros(n_features) for _ in range(n_actions)]

    def theta(self, a: int) -> np.ndarray:
        return np.linalg.solve(self.A[a], self.b[a])

    def scores(self, x: np.ndarray) -> np.ndarray:
        out = np.zeros(self.n_actions)
        for a in range(self.n_actions):
            Ainv = np.linalg.inv(self.A[a])
            mu = float(x @ (Ainv @ self.b[a]))
            bonus = self.alpha * float(np.sqrt(max(x @ Ainv @ x, 0.0)))
            out[a] = mu + bonus
        return out

    def act(self, x: np.ndarray) -> int:
        return int(np.argmax(self.scores(x)))

    def update(self, a: int, x: np.ndarray, r: float) -> None:
        self.A[a] += np.outer(x, x)
        self.b[a] += r * x

    def coefficients(self, feature_names: List[str]) -> pd.DataFrame:
        return pd.DataFrame({ACTIONS[a]: self.theta(a) for a in range(self.n_actions)},
                            index=feature_names).round(3)


X = np.vstack([featurize(r) for r in scored.to_dict("records")])
print("context matrix:", X.shape)

## 15. Baseline policies and training

Baselines are the policies available without learning: accept everything, review everything,
and the current rule-derived routing. The learned policy has to beat the rule-derived
baseline to be worth deploying.

The first `WARMUP_ROUNDS` decisions are drawn uniformly so every action is observed before
the confidence bounds start driving selection. Without it the policy abandons `accept` after
a short run of deniable orders and never recovers it.

In [ ]:
def policy_always_accept(row, x=None):
    return ACTION_INDEX["accept"]


def policy_always_review(row, x=None):
    return ACTION_INDEX["route_review"]


def policy_rule_based(row, x=None):
    det = row["rule_determination"]
    if det == "necessary":
        return ACTION_INDEX["accept"]
    if det == "not_necessary":
        return ACTION_INDEX["route_review"]
    if float(row["mobility_score"]) <= 0:
        return ACTION_INDEX["prompt_mobility"]
    if float(row["monitoring_score"]) <= 0:
        return ACTION_INDEX["prompt_monitoring"]
    return ACTION_INDEX["route_review"]


def evaluate_static_policy(df: pd.DataFrame, fn) -> Dict[str, Any]:
    rows = df.to_dict("records")
    rewards, actions = [], []
    for r in rows:
        a = fn(r)
        actions.append(ACTIONS[a])
        rewards.append(reward(ACTIONS[a], r))
    unsafe = np.mean([(ACTIONS[fn(r)] == "accept") and int(r["would_deny"]) == 1 for r in rows])
    return {"mean_reward": round(float(np.mean(rewards)), 4),
            "unsafe_accept_rate": round(float(unsafe), 4),
            "action_mix": pd.Series(actions).value_counts(normalize=True).round(3).to_dict()}


split = int(len(scored) * 0.7)
train_df = scored.iloc[:split].reset_index(drop=True)
test_df = scored.iloc[split:].reset_index(drop=True)

policy = LinUCBPolicy(n_actions=len(ACTIONS), n_features=len(FEATURE_NAMES), alpha=1.0)

WARMUP_ROUNDS = 200
warmup_rng = np.random.default_rng(SEED)

train_rows = train_df.to_dict("records")
cumulative, running = [], 0.0
for i, r in enumerate(train_rows):
    x = featurize(r)
    a = int(warmup_rng.integers(0, len(ACTIONS))) if i < WARMUP_ROUNDS else policy.act(x)
    rw = reward(ACTIONS[a], r)
    policy.update(a, x, rw)
    running += rw
    cumulative.append(running / (i + 1))

print("training rows:", len(train_rows))
print("running mean reward at 25/50/75/100 percent:",
      [round(cumulative[int(len(cumulative) * p) - 1], 3) for p in (0.25, 0.5, 0.75, 1.0)])

test_rows = test_df.to_dict("records")
learned_rewards, learned_actions = [], []
for r in test_rows:
    a = policy.act(featurize(r))
    learned_actions.append(ACTIONS[a])
    learned_rewards.append(reward(ACTIONS[a], r))

learned_unsafe = np.mean([act == "accept" and int(r["would_deny"]) == 1
                          for act, r in zip(learned_actions, test_rows)])

comparison = pd.DataFrame([
    {"policy": "always_accept", **evaluate_static_policy(test_df, policy_always_accept)},
    {"policy": "always_review", **evaluate_static_policy(test_df, policy_always_review)},
    {"policy": "rule_based_routing", **evaluate_static_policy(test_df, policy_rule_based)},
    {"policy": "linucb_learned",
     "mean_reward": round(float(np.mean(learned_rewards)), 4),
     "unsafe_accept_rate": round(float(learned_unsafe), 4),
     "action_mix": pd.Series(learned_actions).value_counts(normalize=True).round(3).to_dict()},
])
print()
print(comparison.to_string(index=False))
print()
print(policy.coefficients(FEATURE_NAMES).to_string())

## 16. Off-policy evaluation

Once Transport.net logs its own decisions, the learned policy has to be scored against that
log without deploying it. The behaviour policy below stands in for the logged decisions and
records a propensity with every action; replacing `build_logs` with the real decision log
keeps the estimators unchanged.

## 17. Sequential simulation — assumptions

Every number in this section is assumed. Nothing in TripMaster records what an ordering
clinician does after a documentation prompt, so the transition dynamics cannot be estimated
from the data available today. The values below are stated in one place, swept in section 21,
and are the first thing to replace once Transport.net logs prompt and revision events.

The one dynamic that governs whether any of this is worth building: a prompt can only
recover documentation for a condition the patient actually has. `p_latent_support` is the
assumed share of undocumented axes that were true but unwritten. `p_uplift_unsupported` is
the assumed rate at which a prompt produces qualifying-looking text with no clinical basis,
which the reward punishes at the same rate as an outright denial.

In [ ]:
SIMULATION_ONLY = True

ASSUMPTIONS = {
    "p_latent_support": 0.55,
    "p_respond": 0.70,
    "p_uplift_supported": 0.65,
    "p_uplift_unsupported": 0.08,
    "fatigue": 0.65,
    "max_turns": 3,
    "gamma": 0.90,
    "accept_value": 1.0,
    "denial_cost": 6.0,
    "prompt_cost": 0.4,
    "review_deniable": -0.5,
    "review_clean": -1.0,
}

ASSUMPTION_NOTES = {
    "p_latent_support": "Share of undocumented axes the patient genuinely meets.",
    "p_respond": "Probability the ordering clinician answers a prompt at all.",
    "p_uplift_supported": "Given a response and a true condition, chance the axis becomes documented.",
    "p_uplift_unsupported": "Chance a prompt yields qualifying text with no clinical basis.",
    "fatigue": "Multiplier applied to p_respond for each additional prompt in the episode.",
    "max_turns": "Prompts allowed before the episode must terminate.",
    "gamma": "Discount applied to reward one turn later.",
    "accept_value": "Value of accepting an order that will not deny.",
    "denial_cost": "Cost of accepting an order that will deny.",
    "prompt_cost": "Friction cost charged at each prompt.",
    "review_deniable": "Net value of routing an order that would have denied: the denial is "
                       "avoided but labour and delay are spent.",
    "review_clean": "Net value of routing an order that would not have denied: pure waste.",
}

assumption_frame = pd.DataFrame([
    {"parameter": k, "value": v, "note": ASSUMPTION_NOTES[k]} for k, v in ASSUMPTIONS.items()])
print("SIMULATION_ONLY:", SIMULATION_ONLY)
print(assumption_frame.to_string(index=False))

## 18. Episode environment

An episode is one order at entry. State is the documented coverage of each axis plus the
number of prompts already spent, which keeps the table small enough for an SME to read
directly. Coverage levels are derived from the same score cutoffs used everywhere else, so a
level 2 mobility state means the same thing here as a mobility subtotal of 2.0 or more in
section 6.

Prompting an axis already documented to level 2 is not a legal action, so the policy cannot
spend friction on a no-op. Latent support is drawn once per order from `p_latent_support` and
held fixed for the episode. An axis that becomes documented without latent support is recorded as unsupported
and is treated as a denial at accept time.

In [ ]:
def coverage_level(score):
    if score < 0.5:
        return 0
    if score < 2.0:
        return 1
    return 2


LEVEL_POINTS = {0: 0.0, 1: 1.0, 2: 2.5}


def qualifies(mob_level, mon_level):
    if mob_level == 0 and mon_level == 0:
        return False
    total = LEVEL_POINTS[mob_level] + LEVEL_POINTS[mon_level] + 1.0
    return total >= SCORE_NECESSARY_THRESHOLD


def build_episodes(df, assumptions=ASSUMPTIONS, seed=SEED):
    rng = random.Random(seed)
    eps = []
    for r in df.to_dict("records"):
        mob0 = coverage_level(float(r["mobility_score"]))
        mon0 = coverage_level(float(r["monitoring_score"]))
        eps.append({
            "order_id": r["order_id"],
            "mob0": mob0,
            "mon0": mon0,
            "supported": {
                "mobility": True if mob0 > 0 else rng.random() < assumptions["p_latent_support"],
                "monitoring": True if mon0 > 0 else rng.random() < assumptions["p_latent_support"],
            },
            "gold_determination": r["gold_determination"],
            "would_deny": int(r["would_deny"]),
        })
    return eps


STATES = [(m, n, t) for m in range(3) for n in range(3)
          for t in range(ASSUMPTIONS["max_turns"] + 1)]
STATE_INDEX = {s: i for i, s in enumerate(STATES)}


class OrderEpisode:
    def __init__(self, ep, assumptions=ASSUMPTIONS, rng=None):
        self.a = assumptions
        self.rng = rng or random.Random(SEED)
        self.ep = ep
        self.mob = ep["mob0"]
        self.mon = ep["mon0"]
        self.turn = 0
        self.unsupported = False
        self.done = False

    def state(self):
        return (self.mob, self.mon, min(self.turn, self.a["max_turns"]))

    def legal(self):
        acts = ["accept", "route_review"]
        if self.turn >= self.a["max_turns"]:
            return acts
        if self.mob < 2:
            acts.append("prompt_mobility")
        if self.mon < 2:
            acts.append("prompt_monitoring")
        return acts

    def step(self, action):
        a = self.a
        if action == "accept":
            self.done = True
            ok = qualifies(self.mob, self.mon) and not self.unsupported
            return self.state(), (a["accept_value"] if ok else -a["denial_cost"]), True
        if action == "route_review":
            self.done = True
            would_deny = not (qualifies(self.mob, self.mon) and not self.unsupported)
            r = a["review_deniable"] if would_deny else a["review_clean"]
            return self.state(), r, True

        axis = action.replace("prompt_", "")
        level = self.mob if axis == "mobility" else self.mon
        p_respond = a["p_respond"] * (a["fatigue"] ** self.turn)
        reward = -a["prompt_cost"]
        if self.rng.random() < p_respond and level < 2:
            if self.ep["supported"][axis]:
                if self.rng.random() < a["p_uplift_supported"]:
                    level = min(2, level + 1)
            else:
                if self.rng.random() < a["p_uplift_unsupported"]:
                    level = min(2, level + 1)
                    self.unsupported = True
        if axis == "mobility":
            self.mob = level
        else:
            self.mon = level
        self.turn += 1
        if self.turn >= a["max_turns"]:
            return self.state(), reward, False
        return self.state(), reward, False


probe = OrderEpisode(build_episodes(train_df)[0], rng=random.Random(SEED))
print("initial state:", probe.state(), "| legal:", probe.legal())
print("step prompt_mobility ->", probe.step("prompt_mobility"))
print("step accept ->", probe.step("accept"))

## 19. Tabular Q-learning

The state space is 36 cells and the action space is 4, so the learned policy is a table an
SME can read line by line and disagree with. That readability is the reason for choosing
tabular Q-learning over a function approximator; nothing about the problem needs a network
at this size.

This is where the method differs from the contextual bandit in section 15. The bandit scores
a single decision against an immediate reward. Here a prompt changes the state the next
decision is made in, and the discount factor carries the eventual accept or denial back to
the prompt that led to it.

In [ ]:
def train_q(episodes, assumptions=ASSUMPTIONS, n_episodes=40000, alpha=0.15,
            eps_start=1.0, eps_end=0.05, seed=SEED):
    rng = random.Random(seed)
    Q = np.zeros((len(STATES), len(ACTIONS)))
    visits = np.zeros(len(STATES), dtype=int)
    for i in range(n_episodes):
        epsilon = eps_end + (eps_start - eps_end) * max(0.0, 1 - i / (0.7 * n_episodes))
        env = OrderEpisode(rng.choice(episodes), assumptions, rng)
        while not env.done:
            s = STATE_INDEX[env.state()]
            visits[s] += 1
            legal = env.legal()
            legal_idx = [ACTION_INDEX[a] for a in legal]
            if rng.random() < epsilon:
                ai = rng.choice(legal_idx)
            else:
                ai = int(max(legal_idx, key=lambda j: Q[s, j]))
            s2_state, r, done = env.step(ACTIONS[ai])
            s2 = STATE_INDEX[s2_state]
            target = r if done else r + assumptions["gamma"] * float(np.max(Q[s2]))
            Q[s, ai] += alpha * (target - Q[s, ai])
    return Q, visits


train_eps = build_episodes(train_df)
test_eps = build_episodes(test_df, seed=SEED + 1)

Q, VISITS = train_q(train_eps)

q_frame = pd.DataFrame(Q, columns=ACTIONS)
q_frame.insert(0, "turn", [s[2] for s in STATES])
q_frame.insert(0, "monitoring_level", [s[1] for s in STATES])
q_frame.insert(0, "mobility_level", [s[0] for s in STATES])
q_frame["visits"] = VISITS
q_frame["greedy_action"] = [ACTIONS[int(np.argmax(Q[i]))] if VISITS[i] else "unreachable"
                            for i in range(len(STATES))]
print(q_frame[q_frame["turn"] < ASSUMPTIONS["max_turns"]].round(2).to_string(index=False))
print()
print("states never entered:", int((VISITS == 0).sum()), "of", len(STATES))

## 20. Policy comparison under the simulator

Every policy is run through the same environment with the same assumption set, so the
comparison is like for like. The static policies from section 15 are lifted into the
sequential setting by having them re-decide at each turn from the current state.

Unsupported accept rate is reported separately. A policy that raises return by prompting
until something qualifying appears, regardless of whether it is true, will show up there
rather than in the return column.

A learned policy that prompts rarely is a result rather than a failure to train. Under a low
assumed `p_latent_support` most undocumented axes were undocumented because the patient did
not meet them, and prompting spends friction on orders no prompt can rescue.

In [ ]:
def q_policy(env, Q=None):
    Q = Q if Q is not None else globals()["Q"]
    s = STATE_INDEX[env.state()]
    legal = [ACTION_INDEX[a] for a in env.legal()]
    return ACTIONS[int(max(legal, key=lambda j: Q[s, j]))]


def accept_policy(env):
    return "accept"


def review_policy(env):
    return "route_review"


def rule_sequential_policy(env):
    legal = env.legal()
    if env.mob == 0 and "prompt_mobility" in legal:
        return "prompt_mobility"
    if env.mon == 0 and "prompt_monitoring" in legal:
        return "prompt_monitoring"
    return "accept" if qualifies(env.mob, env.mon) else "route_review"


def bandit_sequential_policy(env):
    legal = env.legal()
    if env.turn == 0 and not qualifies(env.mob, env.mon):
        if env.mob == 0 and "prompt_mobility" in legal:
            return "prompt_mobility"
        if "prompt_monitoring" in legal:
            return "prompt_monitoring"
        if "prompt_mobility" in legal:
            return "prompt_mobility"
    return "accept" if qualifies(env.mob, env.mon) else "route_review"


def run_policy(episodes, fn, assumptions=ASSUMPTIONS, n_runs=6, seed=SEED):
    rng = random.Random(seed)
    returns, prompts, unsupported_accepts, terminals = [], [], [], []
    for _ in range(n_runs):
        for ep in episodes:
            env = OrderEpisode(ep, assumptions, rng)
            total, discount, n_prompt = 0.0, 1.0, 0
            last = None
            while not env.done:
                a = fn(env)
                if a.startswith("prompt_"):
                    n_prompt += 1
                _, r, _ = env.step(a)
                total += discount * r
                discount *= assumptions["gamma"]
                last = a
            returns.append(total)
            prompts.append(n_prompt)
            terminals.append(last)
            unsupported_accepts.append(int(last == "accept" and env.unsupported))
    return {"mean_return": round(float(np.mean(returns)), 4),
            "mean_prompts": round(float(np.mean(prompts)), 3),
            "unsupported_accept_rate": round(float(np.mean(unsupported_accepts)), 4),
            "accept_share": round(float(np.mean([t == "accept" for t in terminals])), 3)}


sim_comparison = pd.DataFrame([
    {"policy": "always_accept", **run_policy(test_eps, accept_policy)},
    {"policy": "always_review", **run_policy(test_eps, review_policy)},
    {"policy": "rule_sequential", **run_policy(test_eps, rule_sequential_policy)},
    {"policy": "bandit_style_one_prompt", **run_policy(test_eps, bandit_sequential_policy)},
    {"policy": "q_learned", **run_policy(test_eps, q_policy)},
])
print(sim_comparison.to_string(index=False))

## 21. Sensitivity to the assumptions

The learned policy is only worth deploying where it beats the rule baseline across a
plausible range of clinician behaviour, not at one convenient point. The sweep varies the two
assumptions the result is most exposed to: how often a prompt is answered, and how often the
undocumented axis was true in the first place. The second is the harder constraint — no
prompting strategy can recover documentation for a condition the patient does not have.

Read the `advantage` column as the return the learned policy adds over rule-driven routing at
that assumed responsiveness. Where it is at or below zero, prompting at order entry does not
pay for its own friction and routing to review is the better workflow.

Two readings to check against the numbers rather than assume. First, the advantage is
expected to be widest where `p_latent_support` is low: when most undocumented axes were
undocumented because the patient did not meet them, blanket prompting wastes friction and
selectivity is worth the most. Where nearly every gap is recoverable, the naive rule catches
up and the learned policy adds little. Second, latent support is not part of the state, so the
policy learns an average rather than conditioning on it. That is a limitation of the state
representation, not of the algorithm, and it is the reason `p_latent_support` has to be
measured from the crew PCR rather than tuned.

In [ ]:
SWEEP_RESPOND = [0.3, 0.5, 0.7, 0.9]
SWEEP_LATENT = [0.25, 0.55, 0.85]

sweep_rows = []
for pr in SWEEP_RESPOND:
    for pu in SWEEP_LATENT:
        a = dict(ASSUMPTIONS)
        a["p_respond"] = pr
        a["p_latent_support"] = pu
        eps_tr = build_episodes(train_df, a, seed=SEED)
        eps_te = build_episodes(test_df, a, seed=SEED + 1)
        Qs, _ = train_q(eps_tr, a, n_episodes=15000)
        learned = run_policy(eps_te, lambda e, Qs=Qs: q_policy(e, Qs), a, n_runs=5)
        rule = run_policy(eps_te, rule_sequential_policy, a, n_runs=5)
        sweep_rows.append({
            "p_respond": pr,
            "p_latent_support": pu,
            "q_return": learned["mean_return"],
            "rule_return": rule["mean_return"],
            "advantage": round(learned["mean_return"] - rule["mean_return"], 4),
            "q_prompts": learned["mean_prompts"],
            "q_unsupported_accept": learned["unsupported_accept_rate"],
        })

sweep = pd.DataFrame(sweep_rows)
print(sweep.to_string(index=False))
print()
print("advantage by assumed responsiveness")
print(sweep.pivot(index="p_respond", columns="p_latent_support",
                  values="advantage").round(3).to_string())
print()
print("prompts per order chosen by the learned policy")
print(sweep.pivot(index="p_respond", columns="p_latent_support",
                  values="q_prompts").round(3).to_string())

In [ ]:
def build_logs(df: pd.DataFrame, base_fn, epsilon: float = 0.2, seed: int = SEED):
    rng = np.random.default_rng(seed)
    logs = []
    for r in df.to_dict("records"):
        greedy = base_fn(r)
        if rng.random() < epsilon:
            a = int(rng.integers(0, len(ACTIONS)))
        else:
            a = greedy
        p = (1 - epsilon) * (1.0 if a == greedy else 0.0) + epsilon / len(ACTIONS)
        logs.append({"row": r, "action": a, "propensity": float(p),
                     "reward": reward(ACTIONS[a], r)})
    return logs


def ips_estimate(logs, target_fn, clip: float = 20.0):
    num, weights = [], []
    for e in logs:
        x = featurize(e["row"])
        a_target = target_fn(e["row"], x)
        w = (1.0 / max(e["propensity"], 1e-6)) if a_target == e["action"] else 0.0
        w = min(w, clip)
        num.append(w * e["reward"])
        weights.append(w)
    n = len(logs)
    ips = float(np.sum(num) / n)
    snips = float(np.sum(num) / np.sum(weights)) if np.sum(weights) > 0 else float("nan")
    return {"ips": round(ips, 4), "snips": round(snips, 4),
            "effective_sample": int(np.sum(np.array(weights) > 0))}


logs = build_logs(test_df, policy_rule_based, epsilon=0.2, seed=SEED)

ope = pd.DataFrame([
    {"target_policy": "always_accept",
     **ips_estimate(logs, lambda r, x: policy_always_accept(r))},
    {"target_policy": "always_review",
     **ips_estimate(logs, lambda r, x: policy_always_review(r))},
    {"target_policy": "rule_based_routing",
     **ips_estimate(logs, lambda r, x: policy_rule_based(r))},
    {"target_policy": "linucb_learned",
     **ips_estimate(logs, lambda r, x: policy.act(x))},
])

print("logged mean reward (behaviour policy):",
      round(float(np.mean([e["reward"] for e in logs])), 4))
print()
print(ope.to_string(index=False))

## 22. Outputs

Fourteen frames are written to a single workbook with plain formatting, plus a CSV of the
scored golden records for pivot analysis. When the real golden dataset is in place the same
cell runs unchanged.

In [ ]:
import shutil
from datetime import datetime
from openpyxl import Workbook
from openpyxl.styles import Font

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
xlsx_name = f"med_nec_policy_{stamp}.xlsx"
csv_name = f"med_nec_policy_scored_{stamp}.csv"
local_xlsx = os.path.join(LOCAL_DIR, xlsx_name)
local_csv = os.path.join(LOCAL_DIR, csv_name)

rule_frame = pd.DataFrame(KB["policy_rules"])
reward_frame = pd.DataFrame([{"key": k, "value": v} for k, v in REWARD_TABLE.items()])
coef_frame = policy.coefficients(FEATURE_NAMES).reset_index().rename(columns={"index": "feature"})
gold_mix = golden["gold_source"].value_counts().rename_axis("archetype").reset_index(name="orders")

TABS = [
    ("Sources", source_frame),
    ("Concepts", concept_frame),
    ("Overrides", override_frame if len(override_frame) else pd.DataFrame([{"note": "none"}])),
    ("Policy Rules", rule_frame),
    ("Golden Mix", gold_mix),
    ("Approach Metrics", metrics),
    ("Reward Table", reward_frame),
    ("Policy Comparison", comparison.assign(action_mix=lambda d: d["action_mix"].astype(str))),
    ("Off Policy", ope),
    ("Coefficients", coef_frame),
    ("Assumptions", assumption_frame),
    ("Q Policy", q_frame.round(3)),
    ("Sim Comparison", sim_comparison),
    ("Sensitivity", sweep),
]

wb = Workbook()
wb.remove(wb.active)
for name, frame in TABS:
    ws = wb.create_sheet(name[:31])
    ws.append(list(frame.columns))
    for cell in ws[1]:
        cell.font = Font(name="Calibri", bold=True)
    for rec in frame.itertuples(index=False):
        ws.append(["" if pd.isna(v) else v if isinstance(v, (int, float, str)) else str(v)
                   for v in rec])
    ws.freeze_panes = "A2"
wb.save(local_xlsx)

scored.to_csv(local_csv, index=False)

for local in (local_xlsx, local_csv):
    try:
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        shutil.copyfile(local, os.path.join(OUTPUT_DIR, os.path.basename(local)))
        print("wrote", os.path.join(OUTPUT_DIR, os.path.basename(local)))
    except Exception as e:
        print("workspace copy skipped for", os.path.basename(local), "-", e)
        print("local copy at", local)

## 23. Swap checklist

Facility training content
1. Replace `FACILITY_TRAINING_KNOWLEDGE["concepts"]`, `non_specific_phrases`, and
   `policy_rules` with the extracted content.
2. Set `version` and `pending_ingest = False`.
3. Re-run section 5 and confirm the Overrides table shows only conflicts you accept.

Dave's summary from his discussion with Jen
1. Replace `DAVE_JEN_KNOWLEDGE` the same way.
2. SME guidance outranks facility training, so any weight it declares is applied over the
   training value and logged.
3. Confirm the behavioral concept status and weight before the run that goes to Jen and
   Michelle.

Golden dataset
1. Land the labelled extract at `GOLDEN_TABLE` with the columns in `GOLDEN_SCHEMA`.
2. Change `GOLDEN_SOURCE = SyntheticGoldenSource(...)` to
   `GOLDEN_SOURCE = DeltaTableGoldenSource(GOLDEN_TABLE)`.
3. Re-run from section 11. Sections 12 through 17 need no edit.

Reward function
1. Replace `REWARD_TABLE` values with realized denial cost per order and review labor cost
   per order from the MUSC denial extract.
2. Re-run section 15 and compare against the rule-based routing baseline before any change
   to the live workflow.

Simulation assumptions
1. `ASSUMPTIONS` in section 17 is the only place transition dynamics are declared. Nothing
   downstream hard-codes a probability.
2. `p_respond`, `p_uplift_supported`, and `fatigue` become measurable as soon as Transport.net
   logs a prompt event, the revision that followed it, and the elapsed time between them.
3. `p_latent_support` needs the crew PCR in ImageTrend joined to the order to be measured
   directly, since it asks whether the patient met an axis that the order never documented.
4. Until those land, quote section 21 rather than section 20. The point estimate is one cell
   of the sweep and carries no more weight than the assumption behind it.